# Intro


This notebook works with the GPX data related to a video recording of a camera mounted on a bike. This is combined with an external data source: NWB (Nationaal WegenBestand). Then, information such as type of road, max speed, zebras can be added, which can help looking for the interesting moments. 


Examples with NWB (a.o.):
- Wegvakken toevoegen, met max speed, wegtype (auto's, fietspad, etc), koppelen aan GPX-bestand
- Zebrapad laten weergeven op kaart 

The datasets can be found online:
- Nationaal WegenBestand (general site, including an overview of WegKenmerken): [NWB](https://www.nationaalwegenbestand.nl/nwb-downloaden)
- WegKenmerkenDatabase: [WKD](https://downloads.rijkswaterstaatdata.nl/wkd/)

# Read gpx


In [1]:
import pandas as pd
import geopandas as gpd

from gps_basics import parse_gpx, get_coordinates_and_times
from helper_functions import get_intervals_from_array, intervals_to_time_segments
from nwb_helper_functions import *


In [2]:
# Load gpx, extract data (coordinates and corresponding time in the video)
gpx_file = "route.gpx"
gpx = parse_gpx(gpx_file)
coordinates, times, times_string = get_coordinates_and_times(gpx)

## Example NWB data: add maximum speed of the roads to the GPS

The basis of the visualized maps in this notebook is the shape-file with roads and maximum speeds, downloaded (as a zip) from https://downloads.rijkswaterstaatdata.nl/wkd/Maximum%20Snelheden/. This comes with extra information about all roads, such as road name, municipality, province. 

In [ ]:
# Paths
roads_path = "MaxSnelheid01-12-2024/01-12-2024/Snelheden.shp"
gpx_path = "route.gpx"
output_gpkg = "gpx_with_roads.gpkg"

# 1. Load and filter roads
# (Filter on municipality for performance)
wegen = load_and_filter_roads(roads_path, municipalities = ['Groningen'])	
wegen_cols = ['WVK_ID', 'PVE_CODE', 'GME_ID', 'GME_NAAM', 'STT_NAAM', 'MAXSHD', 'geometry']

# 2. Load GPX points
gpx_points = gpd.read_file(gpx_path, layer="track_points")

# 3. Convert CRS to WGS84
wegen = wegen.to_crs(epsg=4326)
gpx_points = gpx_points.to_crs(epsg=4326)

# 4. Create Folium map
center = gpx_points.geometry.union_all().centroid.coords[:][0][::-1]
m = create_map_with_roads_info(center, wegen, gpx_points, times_string)

# Display map in Jupyter or save
#m.save("interactive_map.html")
m


## Extra data toevoegen: zebrapaden, wegversmallingen en schoolzones

Op basis van csv's ([hier](https://downloads.rijkswaterstaatdata.nl/wkd/) te downloaden). 

In [ ]:
# WegKenMerk: Zebrapaden (VoetgangersOversteekPlaatsen - VOPs)
wkm_df = pd.read_csv('wkd_025-VOP.csv', sep = ';')
naam = 'Zebra'
relevante_cols = ['VAN']

m = create_map_with_roads_info(center, wegen, gpx_points, times_string, marker_info_name = naam, marker_info = wkm_df, marker_info_cols = relevante_cols)
m

In [ ]:
# WegKenMerk: Wegversmalling 
wkm_df = pd.read_csv('wkd_033-WEGVRSMLV2.csv', sep = ';')
naam = 'Wegversmalling'
relevante_cols = ['VAN']

m = create_map_with_roads_info(center, wegen, gpx_points, times_string, marker_info_name = naam, marker_info = wkm_df, marker_info_cols = relevante_cols)
m

In [ ]:
# WegKenMerk: Schoolzones 
wkm_df = pd.read_csv('wkd_027-SCHOOLZNV2.csv', sep = ';')
naam = 'Schoolzone'
relevante_cols = ['VAN', 'TOT']

m = create_map_with_roads_info(center, wegen, gpx_points, times_string, marker_info_name = naam, marker_info = wkm_df, marker_info_cols = relevante_cols)
m

## Example: find the segments where route passes along some WegKenMerk (wkm)

In [ ]:
# Make sure both are GeoDataFrames
wkm_df = pd.read_csv('wkd_025-VOP.csv', sep = ';')
wkm_df = wkm_df.merge(wegen[['WVK_ID', 'geometry']], on='WVK_ID', how='left')

# Remove rows with missing geometry
wkm_df = wkm_df[~wkm_df['geometry'].isna()]

wkm_df = gpd.GeoDataFrame(wkm_df, geometry='geometry')

projected_crs = "EPSG:28992"
wkm_df = wkm_df.to_crs(projected_crs)
gpx_points_proj = gpx_points.to_crs(projected_crs)

# Buffer the points to create small circles around them
buffer_zone = 1 # meters
gpx_points_proj['geometry'] = gpx_points_proj.geometry.buffer(buffer_zone)  

points_in_zones = gpd.sjoin(gpx_points_proj, wkm_df, how='inner', predicate='intersects')

# display time intervals of points in school zones
intervals = get_intervals_from_array(points_in_zones.index)
time_segments = intervals_to_time_segments(intervals, times, to_minutes=True)
print(time_segments)